KPI-1 Encounter Mix by Encounter Class

In [0]:
%sql
create or replace view gold_medical_data.kpi_views.encounter_class_kpi as
with calendar as (
  select
    *
  from
    gold_medical_data.dim_tables.dim_calendar
),
encounters as (
  select
    *
  from
    gold_medical_data.data_cubes.medical_data_cube
)
select
  year(e.encounter_start) as encounter_year,
  c.quater,
  e.encounter_class,
  count(*) as total_encounters,
  round(
    (total_encounters / (sum(count(*)) over (partition by year(e.encounter_start), c.quater))) * 100,2
  ) as total_encounters_percentage
from
  encounters e
    join calendar c
      on month(e.encounter_start) = c.month
group by
  encounter_year,
  c.quater,
  e.encounter_class
order by
  encounter_year,
  c.quater,
  e.encounter_class;

KPI-2 Encounters Over 24 Hours vs Under 24 Hours (%)

In [0]:
%sql
create or replace view gold_medical_data.kpi_views.encounters_durations_kpi as 
with encounter_durations AS (
  select
    date_format(encounter_start, 'yyyy-MM') as encounter_month,
    case
      when
        (unix_timestamp(encounter_stop) - unix_timestamp(encounter_start)) / 3600 > 24
      then
        'More than 24 hours'
      else '24 hours or less'
    end AS duration_category
  from
    gold_medical_data.data_cubes.medical_data_cube
  where
    encounter_start is not null
    and encounter_stop is not null
)
select
  encounter_month,
  duration_category,
  count(*) AS encounter_count,
  round((encounter_count / (sum(count(*)) over (partition by encounter_month)))*100,2) as monthly_percentage
from
  encounter_durations
group by
  encounter_month,
  duration_category
order by
  encounter_month asc,
  duration_category;

KPI-3 Zero Payer Coverage Rate  

In [0]:
%sql
create or replace view gold_medical_data.kpi_views.payer_coverage_kpi as
with encounters as (
  select 
    encounter_id,
    payer_coverage,
    case
      when payer_coverage > 0 then 1
      else 0
    end as has_payer_coverage,
    date_format(encounter_stop, 'yyyy-MM') as encounter_month
  from gold_medical_data.data_cubes.medical_data_cube
)
select
  encounter_month,
  has_payer_coverage,
  count(*) as total_encounters,
  round(
    total_encounters / (sum(count(*)) over (partition by encounter_month)) * 100,
    2
  ) as total_encounters_percentage
from
  encounters
group by
  encounter_month,
  has_payer_coverage
order by
  encounter_month,
  has_payer_coverage;

KPI-4 Top 10 Procedures by Highest Average Base Cost

In [0]:
%sql
create or replace view gold_medical_data.kpi_views.avg_procedure_cost_kpi as
with calendar as (
  select * from
    gold_medical_data.dim_tables.dim_calendar
)
select
  c.half_yearly,
  md.procedure_code,
  round(avg(md.base_procedure_cost), 2) as avg_cost,
  count(*) as total_procedures
from
  gold_medical_data.data_cubes.medical_data_cube md
  join calendar c on month(md.procedure_start) = c.month
group by
  c.half_yearly,
  md.procedure_code
order by
  avg_cost desc,
  total_procedures desc
limit 10;

KPI-5 Average total claim cost for encounters, broken down by payer

In [0]:
%sql
create or replace view gold_medical_data.kpi_views.avg_total_claim_cost_kpi as
select
  payer_id,
  payer_name,
  encounter_class,
  encounter_code,
  round(avg(total_claim_cost),2) as avg_total_claim_cost
from
  gold_medical_data.data_cubes.medical_data_cube
group by
  payer_id,
  payer_name,
  encounter_class,
  encounter_code
order by
  payer_id,
  payer_name,
  encounter_class,
  encounter_code;

KPI-6 30-Day Patient Readmission Rate

In [0]:
%sql
create or replace view gold_medical_data.kpi_views.readmission_rate_kpi as
with encounters as (
  select
    patient_id,
    encounter_start,
    encounter_stop,
    date_format(encounter_start, 'yyyy-MM') as encounter_month,
    lag(encounter_start) over (
      partition by patient_id
      order by encounter_start
    ) as previous_encounter_start,
    lag(encounter_stop) over (
      partition by patient_id
      order by encounter_start
    ) as previous_encounter_stop
  from
    gold_medical_data.data_cubes.medical_data_cube
),
eligible_encounters as (
  select
    encounter_month,
    patient_id,
    encounter_start,
    encounter_stop,
    previous_encounter_stop,
    case
      when previous_encounter_stop is not null
        and encounter_start >= previous_encounter_stop
      then 1
      else 0
    end as is_eligible,
    case
      when previous_encounter_stop is not null
        and encounter_start >= previous_encounter_stop
        and datediff(encounter_start, previous_encounter_stop) <= 30
      then 1
      else 0
    end as is_readmission
  from
    encounters
)
select
  encounter_month as Month,
  sum(is_eligible) as Eligible_Encounters,
  sum(is_readmission) as Readmissions,
  concat(
    round(
      (sum(is_readmission) / nullif(sum(is_eligible),0)) * 100, 2
    ), '%'
  ) as Readmission_Rate
from
  eligible_encounters
group by
  encounter_month
order by
  encounter_month;